In [38]:
from pathlib import Path
import subprocess
from collections import Counter

import pandas as pd

In [39]:
data = Path().resolve() / 'resfinder'

In [40]:
with open(data.parent / "input.txt", "r") as file:
    files = [Path(line.strip()) for line in file.readlines()]

In [41]:
# cmd = f"""
# source activate abricate \
#     && bsub \
#         -o {data / 'abricate.out'} \
#         -e {data / 'abricate.err'} \
#             "abricate \
#                 --db resfinder \
#                 --nopath \
#                 --minid 95 \
#                 --mincov 80 \
#                 -fofn {data.parent / 'input.txt'} \
#                 >> {data / 'resfinder.tab'}
#             "
# """
# subprocess.run(cmd, shell=True)

In [42]:
df = pd.read_table(data / 'resfinder.tab', sep='\t')

In [43]:
cfr_genes = df.groupby('SEQUENCE')['PRODUCT'].apply(lambda x: [y for y in x if 'cfr' in y.lower()]).reset_index(name='cfr_gene').explode('cfr_gene')

In [44]:
cfr_genes = cfr_genes.drop_duplicates()
cfr_genes = cfr_genes.dropna(subset="SEQUENCE")

In [45]:
missing_cfr = cfr_genes[cfr_genes['cfr_gene'].isna()]["SEQUENCE"].to_list()
target_files = [file for file in files if file.stem in missing_cfr or file.stem not in df["SEQUENCE"].to_list()]

In [46]:
# for file in target_files:
#     out = data / 'blast'
#     out.mkdir(exist_ok=True)

#     cmd = f"""
#     source activate blast \
#         && bsub \
#             -o {out / 'blast.out'} \
#             -e {out / 'blast.err'} \
#                 "tblastn \
#                     -task tblastn \
#                     -query {data.parent / 'ARO_3000202-protein.fasta'}\
#                     -subject {file}\
#                     -evalue 1e-5 \
#                     -qcov_hsp_perc 90 \
#                     -max_hsps 1 \
#                     -out {out / f'{file.stem}.csv'} \
#                     -outfmt '20 qseqid sseqid pident qcovhsp evalue'
#                 "
#     """
#     subprocess.run(cmd, shell=True)

In [47]:
def read_blast(file):
    df = pd.read_csv(file)
    
    if df.empty:
        df.loc[0] = None
    
    df["SEQUENCE"] = file.stem
    return df


In [48]:
tblastn = pd.concat([read_blast(file) for file in data.glob("blast/*.csv")], ignore_index=True)

In [49]:
tblastn = tblastn.sort_values("evalue")
tblastn = tblastn.groupby("SEQUENCE").head(1)

In [50]:
tblastn['cfr_gene'] = tblastn['qseqid'].str.split("|").str[-1] + "-like"

In [51]:
cfr_genes = cfr_genes.dropna(subset="cfr_gene")
cfr_genes = pd.concat([cfr_genes, tblastn[["SEQUENCE", "cfr_gene"]]], ignore_index=True)

In [52]:
merged = df.groupby('SEQUENCE')['PRODUCT'].apply(lambda x: ', '.join(sorted(set(x)))).reset_index(name='resfinder_profile')
merged = merged.drop_duplicates()

In [53]:
merged = merged.merge(cfr_genes, on='SEQUENCE', how='outer')

In [54]:
merged.to_csv(data /'resfinder_results.csv', index=False)